# DL500 Exercises -- NLP Preprocessing, Text Representations, and Evaluation

Work through these exercises after completing the DL500 module. Try each exercise before looking at the solutions at the bottom.

**Topics covered:** Text preprocessing pipelines, Bag-of-Words vs TF-IDF vs embeddings, tokenization strategies, BLEU/ROUGE metrics, and clarifying "bagging" vs "Bag-of-Words".

---
## Exercise 1: Text Preprocessing Pipeline

Given the following raw text, apply each preprocessing step manually and describe what it does:

```
"Check out https://example.com for great deals!!! I can't believe it's only $29.99... 
Contact: support@example.com #deal #savings  123-456-7890"
```

Apply in order:
1. URL removal
2. Email removal
3. Contraction expansion
4. Lowercasing
5. Punctuation removal
6. Number removal
7. Whitespace normalization

**Write the result after each step.**

In [ ]:
import sys
sys.path.insert(0, "..")

# Use the project utility
from src.utils.text_preprocessing import normalize_text

raw_text = ("Check out https://example.com for great deals!!! "
            "I can't believe it's only $29.99... "
            "Contact: support@example.com #deal #savings  123-456-7890")

# Apply all preprocessing steps
clean_text = normalize_text(
    raw_text,
    lowercase=True,
    remove_urls=True,
    remove_emails=True,
    expand_contractions=True,
    remove_punctuation=True,
    remove_numbers=True,
)

print(f"Original: {raw_text}")
print(f"\nCleaned:  {clean_text}")

# TODO: Write the intermediate result after each step

---
## Exercise 2: Bag-of-Words vs TF-IDF vs Embeddings

Given these three documents:

- Doc 1: "the cat sat on the mat"
- Doc 2: "the dog sat on the log"
- Doc 3: "the cat chased the dog"

1. Construct the **Bag-of-Words** matrix (term frequency counts)
2. Explain what **TF-IDF** would do differently -- which words get high/low weights?
3. How would **word embeddings** (e.g., Word2Vec) represent these differently from BoW/TF-IDF?
4. Which representation captures that "cat" and "dog" are semantically similar (both animals)?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
]

# 1. Bag-of-Words
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(docs)
print("Bag-of-Words vocabulary:", bow_vectorizer.get_feature_names_out())
print("BoW matrix:")
print(bow_matrix.toarray())

# 2. TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(docs)
print("\nTF-IDF matrix:")
print(np.round(tfidf_matrix.toarray(), 3))

# TODO: Analyze which words get high TF-IDF scores and which get low scores. Why?

---
## Exercise 3: Bagging (Bootstrap Aggregating) vs Bag-of-Words

These two concepts are frequently confused due to the shared word "bag". They are completely unrelated.

**Clarify the difference:**

1. Define **Bag-of-Words (BoW)** -- what domain is it from? What does it represent? What is the "bag" metaphor?

2. Define **Bagging (Bootstrap Aggregating)** -- what domain is it from? What does it do? What is the "bag" metaphor?

3. Could you use both in the same system? Give an example.

4. Fill in the comparison table:

| Aspect | Bag-of-Words | Bagging |
|--------|-------------|--------|
| Domain | ? | ? |
| Purpose | ? | ? |
| Input | ? | ? |
| Output | ? | ? |
| "Bag" refers to | ? | ? |
| Example algorithm | ? | ? |

**Your answers:**

---
## Exercise 4: Tokenization Strategies

Compare different tokenization approaches on the following text:

```
"I'm studying NLP at the state-of-the-art lab. Dr. Smith's email is smith@university.edu."
```

1. **Whitespace tokenization:** Split on spaces only
2. **Basic tokenization:** Split on spaces and separate punctuation
3. **Character-level tokenization:** Each character is a token

For each approach, answer:
- What is the resulting token list?
- What are the advantages and disadvantages?
- What vocabulary size would you expect for a large corpus?

In [ ]:
from src.utils.text_preprocessing import basic_tokenize

text = "I'm studying NLP at the state-of-the-art lab. Dr. Smith's email is smith@university.edu."

# 1. Whitespace tokenization
whitespace_tokens = text.split()
print(f"Whitespace ({len(whitespace_tokens)} tokens):")
print(whitespace_tokens)

# 2. Basic tokenization (our utility)
basic_tokens = basic_tokenize(text, lowercase=True)
print(f"\nBasic ({len(basic_tokens)} tokens):")
print(basic_tokens)

# 3. Character-level
char_tokens = list(text.lower())
print(f"\nCharacter ({len(char_tokens)} tokens):")
print(char_tokens)

# TODO: Discuss pros/cons of each approach

---
## Exercise 5: Building a Vocabulary

Given a small corpus, build a vocabulary and convert texts to integer sequences.

1. Build a vocabulary from the training texts with `min_freq=2`
2. Convert a new test sentence to an integer sequence
3. What happens to words not in the vocabulary?
4. What is the effect of `max_vocab_size` on rare words?

In [ ]:
from src.utils.text_preprocessing import build_vocab, texts_to_sequences

# Training corpus
train_texts = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat and the dog played",
    "the mat is on the floor",
    "the rug is soft and warm",
]

# 1. Build vocabulary with min_freq=2
vocab = build_vocab(train_texts, min_freq=2)
print(f"Vocabulary ({len(vocab)} tokens):")
for token, idx in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"  {idx}: '{token}'")

# 2. Convert a test sentence
test_text = ["the bird sat on the fence"]
sequences = texts_to_sequences(test_text, vocab, max_len=8)
print(f"\nTest text: {test_text[0]}")
print(f"Sequence:  {sequences[0]}")

# 3. Which words became <UNK>?
# TODO: Identify which words in the test text are not in the vocab

---
## Exercise 6: BLEU Score Calculation

BLEU (Bilingual Evaluation Understudy) measures the overlap of n-grams between a candidate translation and reference translation(s).

Given:
- Reference: "the cat is on the mat"
- Candidate A: "the cat is on the mat" (perfect match)
- Candidate B: "the the the the the the" (gaming unigrams)
- Candidate C: "a cat sits on a mat" (paraphrase)

1. Compute the **unigram precision** for each candidate (without brevity penalty)
2. Why does Candidate B get high unigram precision? What mechanism in BLEU addresses this?
3. What is the **brevity penalty** and when does it apply?

In [ ]:
from collections import Counter

reference = "the cat is on the mat".split()
candidate_a = "the cat is on the mat".split()
candidate_b = "the the the the the the".split()
candidate_c = "a cat sits on a mat".split()

def unigram_precision(candidate, reference):
    """Compute simple unigram precision (no clipping)."""
    candidate_counts = Counter(candidate)
    reference_counts = Counter(reference)
    
    # Count matching unigrams
    matches = 0
    for token, count in candidate_counts.items():
        matches += min(count, reference_counts.get(token, 0))  # clipped count
    
    precision = matches / len(candidate)
    return precision, matches, len(candidate)

for name, cand in [("A (perfect)", candidate_a),
                    ("B (gaming)", candidate_b),
                    ("C (paraphrase)", candidate_c)]:
    prec, matches, total = unigram_precision(cand, reference)
    print(f"Candidate {name}: precision={prec:.3f} ({matches}/{total})")

# TODO: Explain the brevity penalty and how BLEU handles candidate B

---
## Exercise 7: ROUGE Score Calculation

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is commonly used for summarization.

Given:
- Reference summary: "the cat sat on the mat in the room"
- System summary A: "the cat sat on the mat"
- System summary B: "the dog ran across the field and jumped"

1. Compute **ROUGE-1** recall, precision, and F1 for both summaries
2. Compute **ROUGE-2** (bigram) recall for summary A
3. When is ROUGE more appropriate than BLEU? Why is ROUGE recall-focused?

In [ ]:
from collections import Counter

reference = "the cat sat on the mat in the room".split()
summary_a = "the cat sat on the mat".split()
summary_b = "the dog ran across the field and jumped".split()

def rouge_1(system, reference):
    """Compute ROUGE-1 precision, recall, F1."""
    sys_counts = Counter(system)
    ref_counts = Counter(reference)
    
    # Overlapping unigrams (clipped)
    overlap = 0
    for token in sys_counts:
        overlap += min(sys_counts[token], ref_counts.get(token, 0))
    
    precision = overlap / len(system) if len(system) > 0 else 0
    recall = overlap / len(reference) if len(reference) > 0 else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
    
    return precision, recall, f1

print("ROUGE-1 Results:")
for name, summary in [("A", summary_a), ("B", summary_b)]:
    p, r, f = rouge_1(summary, reference)
    print(f"  Summary {name}: P={p:.3f}, R={r:.3f}, F1={f:.3f}")

# TODO: Compute ROUGE-2 (bigram) recall for summary A
# Hint: Extract bigrams from both reference and system summary

---
---
# Solutions

Expand each section below to see the solution.

<details>
<summary><b>Solution 1: Text Preprocessing Pipeline</b></summary>

Starting text: `"Check out https://example.com for great deals!!! I can't believe it's only $29.99... Contact: support@example.com #deal #savings  123-456-7890"`

Step-by-step:

1. **URL removal:** `"Check out for great deals!!! I can't believe it's only $29.99... Contact: support@example.com #deal #savings  123-456-7890"`

2. **Email removal:** `"Check out for great deals!!! I can't believe it's only $29.99... Contact: #deal #savings  123-456-7890"`

3. **Contraction expansion:** `"Check out for great deals!!! I cannot believe it is only $29.99... Contact: #deal #savings  123-456-7890"`

4. **Lowercasing:** `"check out for great deals!!! i cannot believe it is only $29.99... contact: #deal #savings  123-456-7890"`

5. **Punctuation removal:** `"check out for great deals i cannot believe it is only 2999 contact deal savings  1234567890"`

6. **Number removal:** `"check out for great deals i cannot believe it is only contact deal savings"`

7. **Whitespace normalization:** `"check out for great deals i cannot believe it is only contact deal savings"`

**Key insight:** The order of operations matters! Removing URLs before lowercasing, and expanding contractions before removing punctuation, ensures correct results.
</details>

<details>
<summary><b>Solution 2: Bag-of-Words vs TF-IDF vs Embeddings</b></summary>

**1. Bag-of-Words matrix:**

| | cat | chased | dog | log | mat | on | sat | the |
|---|---|---|---|---|---|---|---|---|
| Doc 1 | 1 | 0 | 0 | 0 | 1 | 1 | 1 | 2 |
| Doc 2 | 0 | 0 | 1 | 1 | 0 | 1 | 1 | 2 |
| Doc 3 | 1 | 1 | 1 | 0 | 0 | 0 | 0 | 2 |

**2. TF-IDF differences:**
- **"the"** appears in ALL documents: very low IDF, near-zero TF-IDF weight. It is not discriminative.
- **"sat"**, **"on"** appear in 2 of 3 docs: moderate IDF.
- **"mat"**, **"log"**, **"chased"** appear in only 1 doc: highest IDF, most discriminative.
- TF-IDF upweights rare, distinctive words and downweights common ones.

**3. Word embeddings differ by:**
- Producing dense, low-dimensional vectors (e.g., 300 dims vs. vocabulary-sized sparse vectors)
- Capturing semantic similarity: "cat" and "dog" would have similar embeddings because they appear in similar contexts
- Being context-independent (Word2Vec) or context-dependent (BERT)

**4. Only embeddings** capture that "cat" and "dog" are semantically similar. In BoW and TF-IDF, they are orthogonal dimensions with no relationship.
</details>

<details>
<summary><b>Solution 3: Bagging vs Bag-of-Words</b></summary>

These are **completely unrelated concepts** that happen to share the word "bag."

**Bag-of-Words (BoW):**
- **Domain:** Natural Language Processing (NLP)
- **Purpose:** Represent text as a numerical vector for ML models
- **How it works:** Count word frequencies in a document, ignoring word order
- **"Bag" metaphor:** Imagine dumping all the words of a document into a bag -- you can count what's inside but you lose the order

**Bagging (Bootstrap Aggregating):**
- **Domain:** Machine Learning, Ensemble Methods
- **Purpose:** Reduce variance of a model by training multiple models on different subsets of data and averaging their predictions
- **How it works:** Create N bootstrap samples (random sampling with replacement), train one model on each, aggregate predictions (average for regression, majority vote for classification)
- **"Bag" metaphor:** Refers to the Bootstrap AGGregating process; each bootstrap sample is a "bag" of randomly drawn training examples

**Comparison table:**

| Aspect | Bag-of-Words | Bagging |
|--------|-------------|--------|
| Domain | NLP / Text Processing | Ensemble Learning |
| Purpose | Text representation | Variance reduction |
| Input | Raw text documents | Any training dataset |
| Output | Sparse numerical vectors | Ensemble of models |
| "Bag" refers to | Unordered collection of words | Bootstrap sample of data |
| Example algorithm | CountVectorizer, TfidfVectorizer | Random Forest (bagging + decision trees) |

**Using both together:** You could use Bag-of-Words features as input to a Random Forest (which uses bagging). For example: vectorize product reviews with BoW, then train a bagged ensemble of classifiers for sentiment analysis.
</details>

<details>
<summary><b>Solution 4: Tokenization Strategies</b></summary>

**Input:** `"I'm studying NLP at the state-of-the-art lab. Dr. Smith's email is smith@university.edu."`

**1. Whitespace tokenization:**
```
["I'm", "studying", "NLP", "at", "the", "state-of-the-art", "lab.", "Dr.", "Smith's", "email", "is", "smith@university.edu."]
```
- **Pros:** Simple, fast, preserves compound words like "state-of-the-art"
- **Cons:** Punctuation attached to words ("lab."), contractions unsplit ("I'm"), emails treated as single tokens
- **Vocab size:** ~100K-500K for English

**2. Basic tokenization (punctuation-aware):**
```
["i", "'", "m", "studying", "nlp", "at", "the", "state", "-", "of", "-", "the", "-", "art", "lab", ".", "dr", ".", "smith", "'", "s", "email", "is", "smith", "@", "university", ".", "edu", "."]
```
- **Pros:** Clean separation of words and punctuation, handles most cases
- **Cons:** Breaks compound words and contractions into pieces, may lose meaning
- **Vocab size:** ~50K-200K for English

**3. Character-level tokenization:**
```
["i", "'", "m", " ", "s", "t", "u", "d", "y", "i", "n", "g", ...]
```
- **Pros:** No out-of-vocabulary problem, very small vocab (~100 characters), handles any language
- **Cons:** Very long sequences, hard to learn word-level semantics, needs deeper models
- **Vocab size:** ~100-200 (alphabet + digits + punctuation)

**Modern approach:** Subword tokenization (BPE, WordPiece) balances these tradeoffs.
</details>

<details>
<summary><b>Solution 5: Building a Vocabulary</b></summary>

With `min_freq=2`, only tokens appearing 2+ times in the training corpus are included. All others map to `<UNK>` (index 1).

**Vocabulary (min_freq=2):**
```
0: '<PAD>'
1: '<UNK>'
2: 'the'      (appears in all 5 docs)
3: 'on'       (appears in 2 docs)
4: 'sat'      (appears in 2 docs)
5: 'cat'      (appears in 2 docs)
6: 'dog'      (appears in 2 docs)
7: 'mat'      (appears in 2 docs)
8: 'is'       (appears in 2 docs)
9: 'and'      (appears in 2 docs)
10: 'rug'     (appears in 2 docs)
```

Words like "played", "floor", "soft", "warm" appear only once and are excluded.

**Test sentence: "the bird sat on the fence"**
- "the" -> 2
- "bird" -> 1 (UNK -- not in vocab)
- "sat" -> 4
- "on" -> 3
- "the" -> 2
- "fence" -> 1 (UNK -- not in vocab)

Sequence: `[2, 1, 4, 3, 2, 1, 0, 0]` (padded to max_len=8)

**Effect of max_vocab_size:** Limits the vocabulary to the N most frequent tokens. Rare words beyond this limit are mapped to `<UNK>`. This controls model size (embedding layer) and prevents the model from memorizing rare words.
</details>

<details>
<summary><b>Solution 6: BLEU Score Calculation</b></summary>

Reference: "the cat is on the mat" (tokens: the=2, cat=1, is=1, on=1, mat=1)

**Candidate A (perfect match):**
- Unigram precision: 6/6 = 1.000
- Every word matches the reference exactly.

**Candidate B ("the the the the the the"):**
- Naive precision: all 6 words are "the", which appears in reference. Naive: 6/6 = 1.0
- **Clipped precision:** "the" appears 2 times in reference, so we clip to min(6, 2) = 2. Precision: 2/6 = 0.333
- This is how BLEU handles gaming: **modified precision clips** the count of each word to its maximum count in the reference.

**Candidate C ("a cat sits on a mat"):**
- Matches: "cat"(1), "on"(1), "mat"(1) = 3 matches. Clipped precision: 3/6 = 0.500
- "a" and "sits" do not appear in reference.

**Brevity penalty:** Applied when the candidate is shorter than the reference. It prevents systems from producing very short translations that might have high precision. Formula: `BP = exp(1 - ref_len/cand_len)` if `cand_len < ref_len`, else `BP = 1`.

**Full BLEU** also uses 1-gram through 4-gram precision (geometric mean) with the brevity penalty.
</details>

<details>
<summary><b>Solution 7: ROUGE Score Calculation</b></summary>

Reference: "the cat sat on the mat in the room" (9 tokens)

**Summary A: "the cat sat on the mat" (6 tokens)**
- Overlapping tokens: the(2), cat(1), sat(1), on(1), mat(1) = 6 (clipped to reference counts)
- ROUGE-1 Precision: 6/6 = 1.000 (all system tokens match)
- ROUGE-1 Recall: 6/9 = 0.667 (6 of 9 reference tokens covered)
- ROUGE-1 F1: 2 * (1.0 * 0.667) / (1.0 + 0.667) = 0.800

**Summary B: "the dog ran across the field and jumped" (8 tokens)**
- Overlapping tokens: the(2) = 2
- ROUGE-1 Precision: 2/8 = 0.250
- ROUGE-1 Recall: 2/9 = 0.222
- ROUGE-1 F1: 2 * (0.25 * 0.222) / (0.25 + 0.222) = 0.235

**ROUGE-2 (bigram) for Summary A:**
- Reference bigrams: (the, cat), (cat, sat), (sat, on), (on, the), (the, mat), (mat, in), (in, the), (the, room) = 8
- System bigrams: (the, cat), (cat, sat), (sat, on), (on, the), (the, mat) = 5
- Overlapping bigrams: 5
- ROUGE-2 Recall: 5/8 = 0.625

**ROUGE vs BLEU:**
- **ROUGE** is recall-focused: "How much of the reference is captured?" Better for summarization, where you want to ensure all key information is included.
- **BLEU** is precision-focused: "How much of the output is correct?" Better for translation, where you want to ensure the output is accurate.
</details>